In [34]:
import pandas as pd

df = pd.read_csv("Reviews.csv")  # change filename if needed

df = df.sample(10000, random_state=42)
df.shape

(10000, 10)

In [35]:
df = df[['Text', 'Score']]
df.dropna(inplace=True)

df['word_count'] = df['Text'].apply(lambda x: len(x.split()))

df = df[(df['word_count'] >= 50) & (df['word_count'] <= 500)]

In [36]:
def get_sentiment(score):
    if score >= 4:
        return "positive"
    elif score == 3:
        return "neutral"
    else:
        return "negative"

df['sentiment'] = df['Score'].apply(get_sentiment)

In [37]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)        # remove URLs
    text = re.sub(r"[^a-z\s]", "", text)       # remove punctuation & numbers
    words = text.split()
    words = [w for w in words if w not in stop_words]
    return " ".join(words)

df['clean_text'] = df['Text'].apply(clean_text)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\LOQ\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [38]:
from nltk.tokenize import word_tokenize
nltk.download('punkt')

df['tokens'] = df['clean_text'].apply(word_tokenize)

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\LOQ\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [39]:
print(df['sentiment'].value_counts())
df.head()

sentiment
positive    4191
negative     879
neutral      486
Name: count, dtype: int64


,Text,Score,word_count,sentiment,clean_text,tokens
165256,Having tried a couple of other brands of glute...,5,84,positive,tried couple brands glutenfree sandwich cookie...,"[tried, couple, brands, glutenfree, sandwich, ..."
231465,My cat loves these treats. If ever I can't fin...,5,99,positive,cat loves treats ever cant find house pop top ...,"[cat, loves, treats, ever, cant, find, house, ..."
433954,"First there was Frosted Mini-Wheats, in origin...",2,294,negative,first frosted miniwheats original size frosted...,"[first, frosted, miniwheats, original, size, f..."
70260,and I want to congratulate the graphic artist ...,5,122,positive,want congratulate graphic artist putting entir...,"[want, congratulate, graphic, artist, putting,..."
138968,"Previously, I've attempted a recipe with white...",4,278,positive,previously ive attempted recipe white rice noo...,"[previously, ive, attempted, recipe, white, ri..."


In [40]:
from tensorflow.keras.preprocessing.text import Tokenizer

vocab_size = 10000  # limit vocab (important)

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(df['clean_text'])

In [41]:
sequences = tokenizer.texts_to_sequences(df['clean_text'])

print(sequences[0])  # check

[26, 235, 206, 828, 1567, 170, 38, 972, 199, 371, 506, 174, 204, 170, 525, 828, 190, 45, 623, 108, 55, 51, 773, 28, 849, 51, 1417, 3576, 48, 361, 5665, 5, 506, 2692, 6, 68, 193, 828, 206]


In [42]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_length = 100  # good starting point

X = pad_sequences(sequences, maxlen=max_length, padding='post', truncating='post')

In [43]:
label_map = {
    "negative": 0,
    "neutral": 1,
    "positive": 2
}

y = df['sentiment'].map(label_map).values

In [44]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape)
print(y_train.shape)

(4444, 100)
(4444,)


In [45]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.layers import Activation, GlobalMaxPooling1D, BatchNormalization

model = Sequential()

# 1. Embedding
model.add(Embedding(input_dim=10000, output_dim=128, input_length=100))

# 2. LSTM (sequence output for pooling)
model.add(LSTM(64, return_sequences=True))

# 3. Activation (ReLU)
model.add(Activation('relu'))

# 4. Dropout
model.add(Dropout(0.5))

# 5. MaxPooling (global)
model.add(GlobalMaxPooling1D())

# 6. Batch Normalization
model.add(BatchNormalization())

# 7. Dense
model.add(Dense(64, activation='relu'))

# 8. Output
model.add(Dense(3, activation='softmax'))

model.summary()

C:\Users\LOQ\AppData\Roaming\Python\Python311\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_1 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation (Activation)         │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_max_pooling1d            │ ?                      │             0 │
│ (GlobalMaxPooling1D)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [46]:
model.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)


history = model.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/5
139/139 ━━━━━━━━━━━━━━━━━━━━ 11s 51ms/step - accuracy: 0.7268 - loss: 0.7133 - val_accuracy: 0.7743 - val_loss: 0.8633
Epoch 2/5
139/139 ━━━━━━━━━━━━━━━━━━━━ 12s 86ms/step - accuracy: 0.8744 - loss: 0.3457 - val_accuracy: 0.7797 - val_loss: 0.7926
Epoch 3/5
139/139 ━━━━━━━━━━━━━━━━━━━━ 25s 120ms/step - accuracy: 0.9676 - loss: 0.1147 - val_accuracy: 0.7032 - val_loss: 0.8054
Epoch 4/5
139/139 ━━━━━━━━━━━━━━━━━━━━ 11s 80ms/step - accuracy: 0.9869 - loss: 0.0477 - val_accuracy: 0.3147 - val_loss: 1.2944
Epoch 5/5
139/139 ━━━━━━━━━━━━━━━━━━━━ 10s 71ms/step - accuracy: 0.9959 - loss: 0.0173 - val_accuracy: 0.3291 - val_loss: 1.4782


In [47]:
loss, acc = model.evaluate(X_test, y_test)
print("LSTM Test Accuracy:", acc)

In [48]:
y_pred_probs = model.predict(X_test)
y_pred = y_pred_probs.argmax(axis=1)

from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

35/35 ━━━━━━━━━━━━━━━━━━━━ 1s 33ms/step
              precision    recall  f1-score   support

           0       0.35      0.58      0.43       163
           1       0.10      0.69      0.17        88
           2       0.95      0.25      0.39       861

    accuracy                           0.33      1112
   macro avg       0.46      0.50      0.33      1112
weighted avg       0.79      0.33      0.38      1112



In [49]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout
from tensorflow.keras.layers import Activation, BatchNormalization

model_rnn = Sequential()

# 1. Embedding
model_rnn.add(Embedding(input_dim=10000, output_dim=128, input_length=100))

# 2. SimpleRNN
model_rnn.add(SimpleRNN(64))

# 3. Activation (ReLU)
model_rnn.add(Activation('relu'))

# 4. Dropout
model_rnn.add(Dropout(0.5))

# 5. Batch Normalization
model_rnn.add(BatchNormalization())

# 6. Dense
model_rnn.add(Dense(64, activation='relu'))

# 7. Output
model_rnn.add(Dense(3, activation='softmax'))

model_rnn.summary()

C:\Users\LOQ\AppData\Roaming\Python\Python311\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_2 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_1 (Activation)       │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ ?                      │   0 (unbuilt) │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [50]:
model_rnn.compile(
    loss='sparse_categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

history_rnn = model_rnn.fit(
    X_train,
    y_train,
    epochs=5,
    batch_size=32,
    validation_data=(X_test, y_test)
)

Epoch 1/5
139/139 ━━━━━━━━━━━━━━━━━━━━ 14s 51ms/step - accuracy: 0.7059 - loss: 0.8071 - val_accuracy: 0.7743 - val_loss: 0.8353
Epoch 2/5
139/139 ━━━━━━━━━━━━━━━━━━━━ 7s 52ms/step - accuracy: 0.7495 - loss: 0.7484 - val_accuracy: 0.7743 - val_loss: 0.8339
Epoch 3/5
139/139 ━━━━━━━━━━━━━━━━━━━━ 19s 116ms/step - accuracy: 0.7493 - loss: 0.7413 - val_accuracy: 0.7743 - val_loss: 0.8471
Epoch 4/5
139/139 ━━━━━━━━━━━━━━━━━━━━ 20s 114ms/step - accuracy: 0.7495 - loss: 0.7385 - val_accuracy: 0.7743 - val_loss: 0.7301
Epoch 5/5
139/139 ━━━━━━━━━━━━━━━━━━━━ 22s 121ms/step - accuracy: 0.7482 - loss: 0.7431 - val_accuracy: 0.7743 - val_loss: 0.7763


In [51]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

y_pred_rnn = model_rnn.predict(X_test).argmax(axis=1)

accuracy_rnn = accuracy_score(y_test, y_pred_rnn)
precision_rnn = precision_score(y_test, y_pred_rnn, average='weighted')
recall_rnn = recall_score(y_test, y_pred_rnn, average='weighted')
f1_rnn = f1_score(y_test, y_pred_rnn, average='weighted')

print("RNN Accuracy:", accuracy_rnn)
print("RNN Precision:", precision_rnn)
print("RNN Recall:", recall_rnn)
print("RNN F1:", f1_rnn)